# Agentic AI Systems
Explore how AI agents reason, plan, use tools, and maintain memory.
Prerequisites: `pip install numpy matplotlib`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
np.random.seed(42)

## 1. What Is an AI Agent?
An agent perceives its environment, reasons about goals, and takes actions.
Key components: **Perception → Reasoning → Action → Memory**

In [ ]:
# Simple agent architecture
class SimpleAgent:
    """A minimal agent that reasons and acts in a text environment."""
    def __init__(self, name):
        self.name = name
        self.memory = []       # conversation history
        self.tools = {}        # available tools
    
    def register_tool(self, name, func, description):
        self.tools[name] = {'func': func, 'desc': description}
    
    def reason(self, task):
        """Decide which tool to use (rule-based for demo)."""
        task_lower = task.lower()
        for tool_name, tool_info in self.tools.items():
            if any(kw in task_lower for kw in tool_name.split('_')):
                return tool_name
        return None
    
    def act(self, task):
        tool_name = self.reason(task)
        if tool_name:
            result = self.tools[tool_name]['func'](task)
            self.memory.append({'task': task, 'tool': tool_name, 'result': result})
            return f'Used {tool_name}: {result}'
        return 'No suitable tool found.'

# Create agent with tools
agent = SimpleAgent('ResearchBot')
agent.register_tool('calculate', lambda t: eval('2+2'), 'Do math')
agent.register_tool('search', lambda t: 'Found 3 relevant articles', 'Web search')

print(agent.act('Please calculate the sum'))
print(agent.act('Search for recent papers'))
print(f'Memory: {agent.memory}')

## 2. ReAct Pattern: Reason + Act
ReAct interleaves reasoning (thinking) with actions in a loop.

In [ ]:
def react_loop(question, max_steps=5):
    """Simulate the ReAct (Reason + Act) pattern."""
    context = []
    tools = {
        'search': lambda q: f'Results for "{q}": AI agents use LLMs for reasoning.',
        'calculate': lambda q: f'Result: {eval(q) if q.replace(".","").replace("+","").replace("-","").replace("*","").replace("/","").replace(" ","").isdigit() else "N/A"}',
    }
    
    print(f'Question: {question}\n')
    
    # Simulated ReAct steps
    steps = [
        ('Thought', 'I need to search for information about AI agents.'),
        ('Action', 'search("AI agents")'),
        ('Observation', tools['search']('AI agents')),
        ('Thought', 'Now I have enough information to answer.'),
        ('Answer', 'AI agents use LLMs for reasoning and can call tools.'),
    ]
    
    for step_type, content in steps:
        print(f'{step_type}: {content}')

react_loop('What are AI agents?')

## 3. Tool Use Pattern
Agents select and call tools based on the task. Here's a tool dispatcher.

In [ ]:
class ToolDispatcher:
    """Routes tasks to appropriate tools based on keywords."""
    def __init__(self):
        self.tools = {}
        self.call_log = []
    
    def register(self, name, func, keywords):
        self.tools[name] = {'func': func, 'keywords': keywords}
    
    def dispatch(self, query):
        scores = {}
        for name, info in self.tools.items():
            score = sum(1 for kw in info['keywords'] if kw in query.lower())
            scores[name] = score
        
        best_tool = max(scores, key=scores.get)
        if scores[best_tool] == 0:
            return 'No matching tool found'
        
        result = self.tools[best_tool]['func'](query)
        self.call_log.append({'query': query, 'tool': best_tool, 'result': result})
        return f'[{best_tool}] {result}'

dispatcher = ToolDispatcher()
dispatcher.register('weather', lambda q: 'Sunny, 72F', ['weather', 'temperature', 'forecast'])
dispatcher.register('math', lambda q: 'Result: 42', ['calculate', 'math', 'sum', 'multiply'])
dispatcher.register('search', lambda q: '3 articles found', ['search', 'find', 'lookup', 'who', 'what'])

queries = ['What is the weather today?', 'Calculate 6 times 7', 'Search for Python tutorials']
for q in queries:
    print(f'Q: {q}')
    print(f'A: {dispatcher.dispatch(q)}\n')

## 4. Memory Systems
Agents need short-term (conversation) and long-term (persistent) memory.

In [ ]:
class AgentMemory:
    """Demonstrates short-term and long-term memory for agents."""
    def __init__(self, short_term_limit=5):
        self.short_term = deque(maxlen=short_term_limit)  # sliding window
        self.long_term = []   # persistent storage
        self.summary = ''     # compressed memory
    
    def add(self, message):
        self.short_term.append(message)
        # Simulate: if short-term full, compress oldest into summary
        if len(self.short_term) == self.short_term.maxlen:
            self.long_term.append(self.short_term[0])
    
    def get_context(self):
        return {
            'recent': list(self.short_term),
            'long_term_count': len(self.long_term),
        }

mem = AgentMemory(short_term_limit=3)
for i in range(7):
    mem.add(f'Message {i}: User said something about topic {i}')
    ctx = mem.get_context()
    print(f'Step {i}: recent={[m[:15]+"..." for m in ctx["recent"]]}, '
          f'long_term_items={ctx["long_term_count"]}')

## 5. Planning: Task Decomposition
Complex tasks are broken into subtasks. This is key to agent effectiveness.

In [ ]:
def decompose_task(task):
    """Simulate task decomposition (in practice, an LLM does this)."""
    plans = {
        'Write a research report': [
            '1. Search for relevant papers',
            '2. Read and summarize key findings',
            '3. Outline the report structure',
            '4. Write each section',
            '5. Review and edit',
        ],
        'Debug a Python script': [
            '1. Read the error message',
            '2. Locate the failing line',
            '3. Understand the root cause',
            '4. Implement a fix',
            '5. Test the fix',
        ],
    }
    return plans.get(task, ['Analyze task', 'Break into steps', 'Execute', 'Verify'])

for task in ['Write a research report', 'Debug a Python script', 'Unknown task']:
    print(f'\nTask: {task}')
    for step in decompose_task(task):
        print(f'  {step}')

## 6. Multi-Agent Collaboration
Multiple specialized agents can work together on complex tasks.

In [ ]:
class SpecializedAgent:
    def __init__(self, name, specialty):
        self.name = name
        self.specialty = specialty
    
    def process(self, task):
        return f'[{self.name}] Completed {self.specialty} analysis of: {task[:40]}...'

class Orchestrator:
    def __init__(self):
        self.agents = []
    
    def add_agent(self, agent):
        self.agents.append(agent)
    
    def run(self, task):
        print(f'Orchestrator received: {task}\n')
        results = []
        for agent in self.agents:
            result = agent.process(task)
            results.append(result)
            print(result)
        print(f'\nAll {len(self.agents)} agents completed.')
        return results

orch = Orchestrator()
orch.add_agent(SpecializedAgent('Researcher', 'literature review'))
orch.add_agent(SpecializedAgent('Coder', 'code implementation'))
orch.add_agent(SpecializedAgent('Reviewer', 'quality assurance'))
orch.run('Build a sentiment analysis pipeline for product reviews')

## 7. Interview Takeaways
- **Agents** = Perception + Reasoning + Action + Memory
- **ReAct** interleaves thinking and tool-calling in a loop
- **Tool use** is how agents interact with external systems
- **Memory**: short-term (context window) + long-term (vector store/DB)
- **Planning** decomposes complex tasks into manageable subtasks
- **Multi-agent** systems assign specialized roles for collaboration
- Key frameworks: LangChain, LangGraph, CrewAI, AutoGen

---

<small><em>© 2026 AI Nirvana · Disclaimer: Provided as is. No liability assumed.</em></small>